<a href="https://colab.research.google.com/github/pranusha751/ITA0601-machine-learning/blob/main/EX_3(ML).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import math
from collections import Counter

# Step 1: Prepare the standard Play Tennis dataset
# Based on the commonly used Play Tennis dataset
data = {
    'Outlook': [
        'Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain',
        'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain',
        'Sunny', 'Overcast', 'Overcast', 'Rain'
    ],
    'Temperature': [
        'Hot', 'Hot', 'Hot', 'Mild', 'Cool',
        'Cool', 'Cool', 'Mild', 'Cool', 'Mild',
        'Mild', 'Mild', 'Hot', 'Mild'
    ],
    'Humidity': [
        'High', 'High', 'High', 'High', 'Normal',
        'Normal', 'Normal', 'High', 'Normal', 'Normal',
        'Normal', 'High', 'Normal', 'High'
    ],
    'Wind': [
        'Weak', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Strong'
    ],
    'PlayTennis': [
        'No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes',
        'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No'
    ]
}

df = pd.DataFrame(data)
df.to_csv('play_tennis.csv', index=False)

# Step 2: Read CSV file
df = pd.read_csv('play_tennis.csv')

print("TRAINING DATASET:")
print(df.to_string(index=False))


# Step 3: Calculate entropy
def entropy(labels):
    counts = Counter(labels)
    total = len(labels)
    result = 0

    for count in counts.values():
        p = count / total
        result -= p * math.log2(p)

    return result


# Step 4: Calculate information gain
def information_gain(data, attribute, target):
    total_entropy = entropy(data[target])
    weighted_entropy = 0

    for value in data[attribute].unique():
        subset = data[data[attribute] == value]
        weighted_entropy += (
            len(subset) / len(data)
        ) * entropy(subset[target])

    return total_entropy - weighted_entropy


# Step 5: Build the ID3 decision tree
def id3(data, attributes, target):
    labels = data[target]

    # All examples belong to the same class
    if labels.nunique() == 1:
        return labels.iloc[0]

    # No attributes remain
    if not attributes:
        return labels.mode()[0]

    # Select the attribute with highest information gain
    best_attribute = max(
        attributes,
        key=lambda attr: information_gain(data, attr, target)
    )

    tree = {best_attribute: {}}
    remaining = [
        attr for attr in attributes if attr != best_attribute
    ]

    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value]

        if subset.empty:
            tree[best_attribute][value] = labels.mode()[0]
        else:
            tree[best_attribute][value] = id3(
                subset, remaining, target
            )

    return tree


# Step 6: Build and display the tree
features = list(df.columns[:-1])
target = 'PlayTennis'

print("\nINFORMATION GAIN:")
for feature in features:
    print(f"{feature}: {information_gain(df, feature, target):.4f}")

tree = id3(df, features, target)

print("\nGENERATED ID3 DECISION TREE:")
print(tree)


# Step 7: Classify a new sample
def predict(tree, sample, default='No'):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample.get(attribute)

    if value not in tree[attribute]:
        return default

    return predict(tree[attribute][value], sample, default)


new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

prediction = predict(tree, new_sample, df[target].mode()[0])

print("\nNEW SAMPLE:")
print(new_sample)

print("\nPREDICTED CLASS:", prediction)


TRAINING DATASET:
 Outlook Temperature Humidity   Wind PlayTennis
   Sunny         Hot     High   Weak         No
   Sunny         Hot     High Strong         No
Overcast         Hot     High   Weak        Yes
    Rain        Mild     High   Weak        Yes
    Rain        Cool   Normal   Weak        Yes
    Rain        Cool   Normal Strong         No
Overcast        Cool   Normal Strong        Yes
   Sunny        Mild     High   Weak         No
   Sunny        Cool   Normal   Weak        Yes
    Rain        Mild   Normal   Weak        Yes
   Sunny        Mild   Normal Strong        Yes
Overcast        Mild     High Strong        Yes
Overcast         Hot   Normal   Weak        Yes
    Rain        Mild     High Strong         No

INFORMATION GAIN:
Outlook: 0.2467
Temperature: 0.0292
Humidity: 0.1518
Wind: 0.0481

GENERATED ID3 DECISION TREE:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

NEW SA